<a href="https://colab.research.google.com/github/marinaalfarias/ia940/blob/main/Aula11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 11 — Como a máquina aprende uma regra?

**IA/EG940 · 2s2026**

Na Aula 10, **nós** escolhemos a régua: MFCCs, médias, z-score. Hoje, a máquina escolhe. Vamos ver que "aprender" é ajustar parâmetros para reduzir uma perda, que o que se aprende é uma **projeção** do som num espaço de vetores (um *embedding*) e que a regra de treino decide o que esse espaço guarda.

Usamos o **Keras**, a biblioteca de redes neurais que roda sobre o TensorFlow (a mesma base do YAMNet do projeto).

Em vários exercícios aparece **Previsão**. Faça uma cópia do notebook (*Arquivo → Salvar uma cópia no Drive*) e escreva sua resposta na célula livre *antes* de rodar a célula correspondente. Errar a previsão faz parte; o que importa é explicar depois por que errou.

In [ ]:
# Configuração (rode uma vez)
import numpy as np
import matplotlib.pyplot as plt
import librosa
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

SR = 22050
rng = np.random.default_rng(10)      # mesma semente da Aula 10
plt.rcParams['figure.figsize'] = (9, 3.5)

def ouvir(x, sr=SR):
    display(Audio(x, rate=sr))

In [ ]:
import keras
from keras import layers
print('Keras', keras.__version__)

### Os sons da Aula 10

Exatamente a mesma base sintética da Aula 10: 40 sons **tonais**, 40 **percussivos** e 40 **pios**, com ruído de fundo. Cada som vira o vetor de 12 MFCCs médios. Rode e siga em frente.

In [ ]:
def ganho_formantes(f, formantes):
    g = np.full_like(np.asarray(f, float), 1e-3)
    for fc, bw in formantes:
        g = g + 1 / (1 + ((f - fc) / (bw / 2))**2)
    return g

def envoltoria(n, ataque=0.05, decaimento=2.0):
    t = np.arange(n) / SR
    return np.minimum(t / ataque, 1) * np.exp(-t / decaimento)

def normaliza(x):
    return 0.5 * x / np.max(np.abs(x))

def tom(f0, formantes, dur=1.0, ataque=0.05, decaimento=2.0):
    n = int(dur * SR); t = np.arange(n) / SR; x = np.zeros(n)
    for k in range(1, int(SR / 2 / f0)):
        x += ganho_formantes(k * f0, formantes) * np.sin(2*np.pi*k*f0*t + rng.uniform(0, 2*np.pi))
    return normaliza(x * envoltoria(n, ataque, decaimento))

def ruido(formantes, dur=1.0, ataque=0.05, decaimento=2.0):
    n = int(dur * SR)
    X = np.fft.rfft(rng.standard_normal(n)); f = np.fft.rfftfreq(n, 1/SR)
    x = np.fft.irfft(X * ganho_formantes(f, formantes), n)
    return normaliza(x * envoltoria(n, ataque, decaimento))

def pios(dur=1.0):
    n = int(dur * SR); x = np.zeros(n)
    n_pios = rng.integers(2, 6); fb = rng.uniform(1500, 4500); L = int(rng.uniform(0.06, 0.18) * SR)
    for i in range(n_pios):
        ini = int(i * n / n_pios + rng.uniform(0, 0.05) * SR)
        if ini + L > n: break
        t = np.arange(L) / SR
        f = fb + rng.uniform(-1500, 1500) * t / t[-1]
        x[ini:ini+L] += np.sin(2*np.pi*np.cumsum(f)/SR) * np.hanning(L)
    return normaliza(x)

def com_ruido(x, snr_db):
    r = rng.standard_normal(len(x))
    r *= np.sqrt(np.mean(x**2) / np.mean(r**2) / 10**(snr_db/10))
    return x + r

E_base = [(500, 150), (1500, 200), (2500, 250)]
NOMES = ['tonal', 'percussivo', 'pios']
CORES = np.array(['#1f77b4', '#ff7f0e', '#2ca02c'])

def exemplo(classe):
    if classe == 0:
        e = rng.uniform(0.6, 1.6)
        x = tom(rng.uniform(110, 660), [(fc*e, bw*e) for fc, bw in E_base],
                ataque=rng.uniform(0.03, 0.3), decaimento=rng.uniform(0.8, 3))
    elif classe == 1:
        e = rng.uniform(0.4, 3)
        x = ruido([(fc*e, bw*e) for fc, bw in E_base], ataque=0.002, decaimento=rng.uniform(0.03, 0.25))
    else:
        x = pios()
    return com_ruido(x, rng.uniform(0, 20))

def vetor_mfcc(x):
    return librosa.feature.mfcc(y=x, sr=SR, n_mfcc=13)[1:].mean(axis=1)

# Refaz os mesmos sorteios da Aula 10 (sons A, B e C do Ex. 1, depois a base)
for E in ([(500, 150), (1500, 200), (2500, 250)], [(250, 100), (4000, 400), (7000, 600)]):
    com_ruido(tom(220, E), 40)
ruido([(500, 150), (1500, 200), (2500, 250)])

y = np.repeat([0, 1, 2], 40)
sons = [exemplo(c) for c in y]
MF = np.array([vetor_mfcc(x) for x in sons])
print('Base:', MF.shape)

### Ferramentas de visualização

Esta célula define três funções de desenho: um gráfico em que **clicar num ponto toca o som** correspondente, um desenho de redes densas e um diagrama de blocos. Não é preciso ler o código.

In [ ]:
import base64, io, uuid, json
from scipy.io import wavfile
from scipy.signal import resample_poly
from IPython.display import HTML

def _wav_b64(x):
    x = resample_poly(x, 320, 441)                       # 22050 -> 16000 Hz, para o arquivo ficar menor
    x = np.int16(0.8 * x / (np.max(np.abs(x)) + 1e-12) * 32767)
    b = io.BytesIO(); wavfile.write(b, 16000, x)
    return base64.b64encode(b.getvalue()).decode()

def grafico_ouvivel(paineis, sons, rotulos, nomes=NOMES, cores=CORES, lado=320):
    # paineis: lista de (título, E), com E de forma (n_sons, 2). Clique num ponto para ouvir.
    uid = 'g' + uuid.uuid4().hex[:8]
    svgs = []
    for titulo, E in paineis:
        E = np.asarray(E, float)
        lo, hi = np.percentile(E, 1, axis=0), np.percentile(E, 99, axis=0)   # ignora pontos muito afastados
        m = 0.08 * (hi - lo + 1e-12)
        px = 20 + (lado - 40) * (E[:, 0] - lo[0] + m[0]) / (hi[0] - lo[0] + 2 * m[0])
        py = 30 + (lado - 50) * (1 - (E[:, 1] - lo[1] + m[1]) / (hi[1] - lo[1] + 2 * m[1]))
        px, py = np.clip(px, 6, lado - 6), np.clip(py, 26, lado - 6)        # os afastados ficam na borda
        pts = ''.join(
            f'<circle cx="{px[i]:.1f}" cy="{py[i]:.1f}" r="4" fill="{cores[rotulos[i]]}" fill-opacity="0.8" '
            f'data-i="{i}" style="cursor:pointer"><title>som {i} ({nomes[rotulos[i]]})</title></circle>'
            for i in range(len(E)))
        svgs.append(f'<svg width="{lado}" height="{lado}" style="border:1px solid #ccc;background:#fff">'
                    f'<text x="{lado/2}" y="18" text-anchor="middle" font-size="13" font-family="sans-serif">{titulo}</text>'
                    f'{pts}</svg>')
    legenda = ' '.join(f'<span style="color:{cores[c]}">&#9679;</span> {nomes[c]}' for c in range(len(nomes)))
    audios = json.dumps([_wav_b64(x) for x in sons])
    script = '''
<script>
(function(){
  const A = __AUDIOS__, lab = __LAB__, nomes = __NOMES__;
  const raiz = document.getElementById('__UID__');
  const info = document.getElementById('__UID___info');
  const pts = raiz.querySelectorAll('circle');
  let hist = [];
  pts.forEach(c => c.addEventListener('click', () => {
    const i = +c.dataset.i;
    raiz.querySelectorAll('circle[data-i="' + i + '"]').forEach(q => q.parentNode.appendChild(q));
    new Audio('data:audio/wav;base64,' + A[i]).play();
    hist = [i].concat(hist.filter(j => j !== i)).slice(0, 2);
    pts.forEach(p => {
      const j = +p.dataset.i;
      p.setAttribute('r', hist.includes(j) ? 7 : 4);
      p.setAttribute('stroke', j === hist[0] ? 'black' : (j === hist[1] ? 'gray' : 'none'));
      p.setAttribute('stroke-width', 2);
    });
    info.textContent = 'Agora: som ' + i + ' (' + nomes[lab[i]] + ')' +
      (hist.length > 1 ? '   |   antes: som ' + hist[1] + ' (' + nomes[lab[hist[1]]] + ')' : '');
  }));
})();
</script>'''
    script = (script.replace('__AUDIOS__', audios).replace('__LAB__', json.dumps([int(r) for r in rotulos]))
                    .replace('__NOMES__', json.dumps(list(nomes))).replace('__UID__', uid))
    html = (f'<div id="{uid}" style="font-family:sans-serif;font-size:13px">'
            f'<div style="display:flex;gap:12px;flex-wrap:wrap">{"".join(svgs)}</div>'
            f'<div style="margin-top:6px">{legenda}</div>'
            f'<div id="{uid}_info" style="margin-top:4px;color:#444">Clique num ponto para ouvir o som. '
            f'O mesmo som fica destacado em todos os painéis.</div></div>' + script)
    display(HTML(html))

def desenha_dense(tamanhos, nomes_camadas, titulo=''):
    # Desenha uma rede densa: cada neurônio de uma camada liga-se a todos da camada seguinte.
    fig, ax = plt.subplots(figsize=(1.8 * len(tamanhos) + 1, 4))
    pos = []
    for k, n in enumerate(tamanhos):
        ys = np.linspace(0, 1, n) if n > 1 else np.array([0.5])
        pos.append([(k, yy) for yy in ys])
    for k in range(len(tamanhos) - 1):
        for (x0, y0) in pos[k]:
            for (x1, y1) in pos[k + 1]:
                ax.plot([x0, x1], [y0, y1], color='gray', lw=0.4, alpha=0.6)
    for k, camada in enumerate(pos):
        ax.scatter(*zip(*camada), s=90, c='white', edgecolors='k', zorder=3)
        ax.text(k, -0.12, nomes_camadas[k], ha='center', va='top', fontsize=9)
    ax.set_xlim(-0.6, len(tamanhos) - 0.4); ax.set_ylim(-0.45, 1.1); ax.axis('off')
    ax.set_title(titulo, fontsize=10); plt.tight_layout(); plt.show()

def desenha_blocos(blocos, titulo=''):
    # blocos: lista de (nome, forma da saída); desenha da esquerda para a direita.
    fig, ax = plt.subplots(figsize=(2.3 * len(blocos), 1.8))
    for k, (nome, forma) in enumerate(blocos):
        ax.add_patch(plt.Rectangle((k * 2.3, 0), 1.9, 1, fc='#eef3fb', ec='k'))
        ax.text(k * 2.3 + 0.95, 0.62, nome, ha='center', va='center', fontsize=9)
        ax.text(k * 2.3 + 0.95, 0.25, forma, ha='center', va='center', fontsize=8, color='#555')
        if k < len(blocos) - 1:
            ax.annotate('', xy=(k * 2.3 + 2.3, 0.5), xytext=(k * 2.3 + 1.9, 0.5),
                        arrowprops=dict(arrowstyle='->'))
    ax.set_xlim(-0.2, 2.3 * len(blocos)); ax.set_ylim(-0.2, 1.3); ax.axis('off')
    ax.set_title(titulo, fontsize=10); plt.tight_layout(); plt.show()

---
## Exercício 1 — Uma projeção aprendida

Queremos ver os 12 MFCCs em um plano. A maneira clássica é o **PCA** (análise de componentes principais), uma **fórmula** fechada: ele encontra as duas direções em que os dados mais variam e projeta cada ponto sobre elas. Agora, uma rede vai **aprender** a projeção.

### A camada densa

O bloco básico do Keras é a camada `Dense(n)`: cada um dos seus *n* neurônios recebe **todas** as entradas, multiplica cada uma por um peso, soma tudo e acrescenta um viés:

$$h_j = \sum_{i=1}^{n_{\text{entrada}}} w_{ji}\, x_i + b_j, \qquad j = 1, \dots, n$$

Em forma matricial, $\mathbf{h} = W\mathbf{x} + \mathbf{b}$. Os pesos $w_{ji}$ e os vieses $b_j$ são os **parâmetros**: uma camada `Dense(n)` com $n_{\text{entrada}}$ entradas tem $n_{\text{entrada}} \times n$ pesos (um por ligação) e $n$ vieses (um por neurônio). Opcionalmente, o resultado passa por uma função não linear, a **ativação**. Sem ativação, a camada é linear.

### O autoencoder

Nossa rede tem duas camadas densas lineares. A primeira (o **gargalo**) leva os 12 números a 2; a segunda tenta reconstruir os 12 a partir dos 2. Uma rede que aprende a reconstruir a própria entrada passando por um gargalo se chama **autoencoder**.

A **regra** é dada por uma perda: o erro quadrático médio entre a entrada e a reconstrução. Treinar é repetir muitas vezes (épocas): calcular a perda, calcular em que direção cada parâmetro deve mudar para ela diminuir (o **gradiente**) e dar um pequeno passo nessa direção.

In [ ]:
desenha_dense([12, 2, 12], ['entrada\n12 MFCCs', 'gargalo\nDense(2)', 'saída\nDense(12)'],
              'Autoencoder linear: cada linha cinza é um peso')

**Previsão:** quantos parâmetros essa rede tem? E o plano aprendido por ela será igual ao do PCA?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
Z = StandardScaler().fit_transform(MF)        # cada MFCC com média 0 e desvio 1

keras.utils.set_random_seed(0)
entrada = keras.Input(shape=(12,))
gargalo = layers.Dense(2, name='gargalo')(entrada)      # 12 -> 2, sem ativação (linear)
saida   = layers.Dense(12, name='saida')(gargalo)       # 2 -> 12, sem ativação (linear)
autoencoder = keras.Model(entrada, saida)
autoencoder.compile(optimizer=keras.optimizers.Adam(0.01), loss='mse')
autoencoder.summary()                                   # a coluna "Param #" conta os parâmetros

hist = autoencoder.fit(Z, Z, epochs=200, verbose=0)    # a entrada é também o alvo

plt.plot(hist.history['loss'])
plt.xlabel('época'); plt.ylabel('perda (erro quadrático médio)')
plt.title('Treino do autoencoder linear'); plt.grid(alpha=0.3); plt.show()

In [ ]:
codificador = keras.Model(entrada, gargalo)             # só a metade de ida: 12 -> 2
E_ae = codificador.predict(Z, verbose=0)

pca = PCA(n_components=2).fit(Z)
E_pca = pca.transform(Z)
erro_pca = np.mean((Z - pca.inverse_transform(E_pca))**2)
erro_ae = np.mean((Z - autoencoder.predict(Z, verbose=0))**2)

grafico_ouvivel([(f'PCA (erro {erro_pca:.3f})', E_pca), (f'Autoencoder (erro {erro_ae:.3f})', E_ae)], sons, y)

Clique em dois pontos próximos e ouça: eles soam parecidos? Clique num ponto e veja onde o mesmo som está no outro painel.

### 1b. É o mesmo plano?

Se o autoencoder encontrou o mesmo plano do PCA, deve existir uma transformação linear (rotação, escala, espelhamento) que leva as coordenadas de um nas do outro. Testamos isso com uma regressão linear: $R^2 = 1$ significa que uma converte-se exatamente na outra.

In [ ]:
from sklearn.linear_model import LinearRegression
r2 = LinearRegression().fit(E_ae, E_pca).score(E_ae, E_pca)
print(f'R² (coordenadas do PCA a partir das do autoencoder) = {r2:.3f}')

**Para discutir:** de onde vem cada parâmetro da tabela do `summary()`? Se a rede é linear, como o PCA, por que as duas figuras não são idênticas? Qual foi a regra, e quem a escolheu?

---
## Exercício 2 — A regra define a régua

Agora cada camada ganha uma ativação não linear, a tangente hiperbólica (`tanh`), e treinamos **o mesmo codificador** com duas regras diferentes, usando **só os sons tonais e percussivos**. Os pios ficam de fora do treino.

- **Regra 1, reconstruir:** o gargalo deve guardar o suficiente para refazer os 12 MFCCs. Não usa rótulos.
- **Regra 2, classificar:** depois do gargalo vem uma camada que diz "tonal" ou "percussivo". Usa rótulos.

Depois do treino, os **120 sons**, incluindo os pios que nenhuma das redes viu, passam pelos dois codificadores. A saída do gargalo, com 2 números, é o **embedding** de cada som.

In [ ]:
desenha_dense([12, 8, 2, 8, 12], ['entrada', 'Dense(8)\ntanh', 'gargalo\nDense(2)\ntanh', 'Dense(8)\ntanh', 'saída\nDense(12)'],
              'Regra 1: reconstruir')
desenha_dense([12, 8, 2, 2], ['entrada', 'Dense(8)\ntanh', 'gargalo\nDense(2)\ntanh', 'saída\nDense(2)\nsoftmax'],
              'Regra 2: classificar (a saída dá a probabilidade de cada classe)')

**Previsão:** em cada embedding, onde vão cair os pios?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
treino = y < 2                                          # só tonal e percussivo
sc = StandardScaler().fit(MF[treino])
Z = sc.transform(MF)

def codificador_novo():
    ent = keras.Input(shape=(12,))
    h = layers.Dense(8, activation='tanh')(ent)
    g = layers.Dense(2, activation='tanh', name='gargalo')(h)
    return ent, g

# Regra 1: reconstruir
keras.utils.set_random_seed(1)
ent, g = codificador_novo()
saida = layers.Dense(12)(layers.Dense(8, activation='tanh')(g))
rede_rec = keras.Model(ent, saida)
rede_rec.compile(optimizer=keras.optimizers.Adam(0.01), loss='mse')
rede_rec.fit(Z[treino], Z[treino], epochs=500, verbose=0)
cod_rec = keras.Model(ent, g)

# Regra 2: classificar tonal x percussivo
keras.utils.set_random_seed(1)
ent, g = codificador_novo()
saida = layers.Dense(2, activation='softmax')(g)
rede_cla = keras.Model(ent, saida)
rede_cla.compile(optimizer=keras.optimizers.Adam(0.01), loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])
rede_cla.fit(Z[treino], y[treino], epochs=300, verbose=0)
cod_cla = keras.Model(ent, g)

E_rec = cod_rec.predict(Z, verbose=0)
E_cla = cod_cla.predict(Z, verbose=0)
grafico_ouvivel([('Regra 1: reconstruir', E_rec), ('Regra 2: classificar', E_cla)], sons, y)

### 2b. O que os eixos da regra 1 representam?

A regra 1 não conhece as classes, então não há motivo para seus eixos separarem classes. Eles guardam as maiores variações dos MFCCs, sejam elas quais forem. Para descobrir que variações são essas, ouça pontos ao longo de cada eixo no gráfico acima, e compare com a figura abaixo, em que cada som é colorido pelo seu **centroide espectral** (o "brilho", da Aula 5).

In [ ]:
centroide = np.array([librosa.feature.spectral_centroid(y=x, sr=SR).mean() for x in sons])
plt.figure(figsize=(5, 4))
plt.scatter(*E_rec.T, c=centroide, cmap='viridis', s=18)
plt.colorbar(label='centroide espectral (Hz)')
plt.xlabel('eixo 1'); plt.ylabel('eixo 2'); plt.title('Regra 1, colorida pelo brilho', fontsize=10)
plt.tight_layout(); plt.show()
for j in range(2):
    print(f'correlação entre o eixo {j+1} e o centroide: {np.corrcoef(E_rec[:, j], centroide)[0, 1]:+.2f}')

### 2c. O embedding serve para uma tarefa nova?

As duas redes foram treinadas para uma tarefa (tonal contra percussivo). Usamos agora seus embeddings para outra: separar as **três** classes com um kNN, com 10 divisões treino/teste. Para ver onde estão os acertos e os erros, medimos a fração reconhecida de **cada** classe.

**Previsão:** qual representação reconhece melhor cada classe?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
def knn_por_classe(E, repeticoes=10):
    rec = np.zeros((repeticoes, 3))
    for s in range(repeticoes):
        a, b, ya, yb = train_test_split(E, y, test_size=0.3, stratify=y, random_state=s)
        sc_ = StandardScaler().fit(a)
        pred = KNeighborsClassifier(5).fit(sc_.transform(a), ya).predict(sc_.transform(b))
        rec[s] = [np.mean(pred[yb == c] == c) for c in range(3)]
    return rec.mean(axis=0)

print(f'{"":32s} {"tonal":>7s} {"percussivo":>11s} {"pios":>6s} {"média":>7s}')
for nome, E in [('MFCC médio (12 dim.)', MF), ('Regra 1: reconstruir (2 dim.)', E_rec),
                ('Regra 2: classificar (2 dim.)', E_cla)]:
    r = knn_por_classe(E)
    print(f'{nome:32s} {r[0]:7.0%} {r[1]:11.0%} {r[2]:6.0%} {r.mean():7.0%}')

**Para discutir:** os rótulos de treino mudaram o lugar dos pontos? Por quê? O que cada regra obrigou o gargalo a guardar, e o que deixou livre para descartar? Um embedding que é ótimo para a tarefa em que foi treinado é necessariamente bom para outra?

---
## Exercício 3 — A média vira camada

Voltamos ao problema do Ex. 5 da Aula 10: o golpe e o golpe invertido têm o mesmo MFCC médio. Desta vez, a rede recebe **a sequência de quadros de MFCC** (44 quadros × 13 coeficientes, incluindo o coeficiente 0), ou seja, a mesma informação a partir da qual calculamos a média. A diferença é que agora a rede decide o que fazer com os quadros **antes** da média.

Geramos 60 golpes e 60 golpes invertidos, com envoltória espectral, decaimento e SNR sorteados.

In [ ]:
def golpe_aleatorio():
    e = rng.uniform(0.5, 2)
    x = ruido([(fc*e, bw*e) for fc, bw in E_base], ataque=0.002, decaimento=rng.uniform(0.05, 0.3))
    return com_ruido(x, rng.uniform(10, 30))

def quadros(x):
    return librosa.feature.mfcc(y=x, sr=SR, n_mfcc=13).T          # (quadros, 13)

N = 60
golpes = [golpe_aleatorio() for _ in range(N)]
invertidos = [golpe_aleatorio()[::-1] for _ in range(N)]
Q = np.array([quadros(x) for x in golpes + invertidos])
yg = np.r_[np.zeros(N), np.ones(N)]                                # 0 = golpe, 1 = invertido
T = Q.shape[1]
print('Entrada da rede:', Q.shape, '(sons, quadros, coeficientes)')

t_q = np.arange(T) * 512 / SR                                    # instante de cada quadro (salto de 512 amostras)
fig, eixos = plt.subplots(2, 2, figsize=(11, 5), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
Ms = [quadros(x).T for x in (golpes[0], golpes[0][::-1])]          # um golpe e a sua inversão
lim = max(np.abs(M[1:]).max() for M in Ms)                          # mesma escala de cor nos dois
for j, (nome, M) in enumerate(zip(['Golpe', 'O mesmo golpe invertido'], Ms)):
    eixos[0, j].imshow(M[1:], origin='lower', aspect='auto', cmap='coolwarm', vmin=-lim, vmax=lim,
                       extent=[0, t_q[-1], 0.5, 12.5])
    eixos[0, j].set_title(f'{nome}: MFCC 1 a 12 ao longo do tempo'); eixos[0, j].set_ylabel('coeficiente')
    eixos[1, j].plot(t_q, M[0], 'k'); eixos[1, j].set_ylabel('MFCC 0 (energia)')
    eixos[1, j].set_xlabel('tempo (s)'); eixos[1, j].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Golpe'); ouvir(golpes[0])
print('O mesmo golpe invertido'); ouvir(golpes[0][::-1])

Comparamos duas redes. As duas terminam com a **média no tempo** (`GlobalAveragePooling1D`), cuja saída é o embedding do som, seguida de um neurônio que diz "golpe" ou "invertido".

- **Rede A, quadro a quadro:** antes da média, a mesma camada `Dense(32)` é aplicada a **cada quadro isoladamente**.
- **Rede B, convolucional:** antes da média, uma camada `Conv1D` aplica 32 filtros que olham **9 quadros vizinhos** de cada vez e deslizam ao longo do tempo. Cada filtro é um filtro FIR, como os da Aula 2, mas com coeficientes aprendidos.

Na rede B, a forma muda de (44, 13) para (36, 32) por dois motivos diferentes. **No tempo**, o filtro só é calculado onde cabe inteiro dentro do sinal (convolução "válida", como `np.convolve(..., mode='valid')`): sobram 44 − 9 + 1 = 36 instantes. **Nos canais**, cada um dos 32 filtros é um bloco de 9 quadros × 13 coeficientes, que combina todos os coeficientes e produz **um** número por instante; 32 filtros dão 32 canais. É um banco de filtros FIR multicanal, com 9 × 13 × 32 pesos e 32 vieses.

In [ ]:
desenha_blocos([('entrada', f'({T}, 13)'), ('Dense(32), ReLU\nem cada quadro', f'({T}, 32)'),
                ('média no tempo\n= embedding', '(32,)'), ('Dense(1)\nsigmoide', 'golpe ou invertido')],
               'Rede A: quadro a quadro + média')
desenha_blocos([('entrada', f'({T}, 13)'), ('Conv1D: 32 filtros\nde 9 quadros, ReLU', f'({T - 8}, 32)'),
                ('média no tempo\n= embedding', '(32,)'), ('Dense(1)\nsigmoide', 'golpe ou invertido')],
               'Rede B: convolução + média')

**Previsão:** qual das duas redes separa o golpe do invertido? A rede A conseguiria, com mais neurônios ou mais épocas?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
def rede(tipo, semente, L=9):
    keras.utils.set_random_seed(semente)
    ent = keras.Input(shape=(T, 13))
    if tipo == 'A':
        h = layers.Dense(32, activation='relu')(ent)                   # o mesmo em cada quadro
    else:
        h = layers.Conv1D(32, kernel_size=L, activation='relu')(ent)   # 32 filtros de L quadros
    emb = layers.GlobalAveragePooling1D(name='embedding')(h)          # a média no tempo
    saida = layers.Dense(1, activation='sigmoid')(emb)
    m = keras.Model(ent, saida)
    m.compile(optimizer=keras.optimizers.Adam(0.01), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def treina_e_testa(tipo, semente, L=9):
    a, b, ya, yb = train_test_split(Q, yg, test_size=0.3, stratify=yg, random_state=semente)
    mu, dp = a.mean(axis=(0, 1)), a.std(axis=(0, 1))                   # z-score só com o treino
    m = rede(tipo, semente, L)
    m.fit((a - mu) / dp, ya, epochs=100, verbose=0)
    return m, m.evaluate((b - mu) / dp, yb, verbose=0)[1], (mu, dp)

rede_A, acc_A, norm_A = treina_e_testa('A', 0)
rede_B, acc_B, norm_B = treina_e_testa('B', 0)
rede_A.summary(); rede_B.summary()
print(f'Rede A (quadro a quadro + média): acurácia de teste = {acc_A:.0%}')
print(f'Rede B (convolução + média):      acurácia de teste = {acc_B:.0%}')

### 3b. A régua aprendida

O embedding de cada rede é a saída da média (32 números). Medimos nele as mesmas distâncias do Ex. 5 da Aula 10, com um golpe novo, sua inversão e outra realização do mesmo golpe.

In [ ]:
def embedding(m, norm, x):
    mu, dp = norm
    cod = keras.Model(m.input, m.get_layer('embedding').output)
    return cod.predict(((quadros(x) - mu) / dp)[None], verbose=0)[0]

g = com_ruido(ruido(E_base, ataque=0.002, decaimento=0.15), 30)
outro = com_ruido(ruido(E_base, ataque=0.002, decaimento=0.15), 30)
for nome, m, norm in [('Rede A', rede_A, norm_A), ('Rede B', rede_B, norm_B)]:
    eg, ei, eo = embedding(m, norm, g), embedding(m, norm, g[::-1]), embedding(m, norm, outro)
    print(f'{nome}:  golpe–invertido = {np.linalg.norm(eg - ei):.2f}    golpe–outro golpe = {np.linalg.norm(eg - eo):.2f}')

### 3c. Quanto tempo o filtro precisa enxergar?

Cada grupo escolhe um tamanho de filtro $L$ (em quadros) e treina a rede B três vezes com ele. Os resultados dos grupos formam, juntos, uma curva de acurácia contra a duração do filtro.

**Previsão:** que acurácia a rede B terá com o seu $L$? E com $L = 1$, o que a rede B passa a ser?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
MEU_L = 2                      # <- cada grupo escolhe: 1, 2, 3, 9 quadros

accs = [treina_e_testa('B', s, L=MEU_L)[1] for s in range(3)]
print(f'filtro de {MEU_L} quadros (~{MEU_L * 512 / SR * 1000:.0f} ms): '
      f'acurácia média = {np.mean(accs):.0%}   (rodadas: {np.round(accs, 2)})')

### 3d. Aprendeu de verdade, ou foi sorte?

O treino de uma rede é aleatório: os pesos iniciais são sorteados, e a divisão treino/teste também. Uma única rodada não prova nada. Cada grupo compara **dois tamanhos de filtro**, $L_1$ e $L_2$, treinando cada um 10 vezes com sementes diferentes, e conta em quantas rodadas o $L_2$ vence.

**Previsão:** em quantas das 10 rodadas o seu $L_2$ vence o $L_1$?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
L1, L2 = 1, 2                  # <- o par do seu grupo

acc_1, acc_2 = [], []
for s in range(10):
    acc_1.append(treina_e_testa('B', s, L=L1)[1])
    acc_2.append(treina_e_testa('B', s, L=L2)[1])
acc_1, acc_2 = np.array(acc_1), np.array(acc_2)

vitorias, empates = np.sum(acc_2 > acc_1), np.sum(acc_2 == acc_1)
print(f'L = {L1}:', np.round(acc_1, 2))
print(f'L = {L2}:', np.round(acc_2, 2))
print(f'L = {L2} venceu {vitorias} vezes; {empates} empates; '
      f'diferença média = {np.mean(acc_2 - acc_1):+.1%} ± {np.std(acc_2 - acc_1):.1%}')

**Hipótese nula (H0):** os dois tamanhos de filtro são equivalentes; em cada rodada, o $L_2$ vencer é como tirar cara numa moeda honesta.

**Hipótese alternativa (H1):** o $L_2$ é melhor.

Se H0 fosse verdadeira, com que frequência veríamos tantas vitórias, ou mais? Respondemos jogando moedas no computador: 100 000 vezes, jogamos uma moeda por rodada e contamos as caras. O gráfico mostra **com que frequência cada número de vitórias aconteceria se H0 fosse verdadeira**. O **p-valor** é a soma das barras vermelhas: a fração dos sorteios com tantas vitórias quanto as observadas, ou mais. Os empates são descartados, porque não favorecem nenhuma das hipóteses.

In [ ]:
def p_valor(vitorias, n, sorteios=100_000, mostrar=True):
    if n == 0:
        print('Todas as rodadas empataram: os dados não permitem decidir.'); return np.nan
    caras = rng.integers(0, 2, size=(sorteios, n)).sum(axis=1)       # vitórias, se H0 for verdadeira
    p = np.mean(caras >= vitorias)
    if mostrar:
        freq = np.bincount(caras, minlength=n + 1) / sorteios
        k = np.arange(n + 1)
        plt.bar(k, freq, color=np.where(k >= vitorias, 'tab:red', 'lightgray'), edgecolor='k')
        plt.xticks(k)
        plt.xlabel(f'número de vitórias em {n} rodadas, se H0 for verdadeira')
        plt.ylabel('frequência nos sorteios')
        plt.title(f'Observado: {vitorias} vitórias.  p-valor = soma das barras vermelhas = {p:.3f}', fontsize=10)
        plt.tight_layout(); plt.show()
    return p

In [ ]:
p_valor(vitorias, 10 - empates)

### 3e. Um caso da Aula 10 (se houver tempo)

Na Aula 10, o kNN com $k = 11$ pareceu melhor que com $k = 1$. Aplicamos o mesmo raciocínio, com 10 divisões, usando os MFCC 1 e 3 da mesma base.

**Previsão:** o p-valor será maior ou menor que o do seu grupo?

✏️ **Sua previsão:** *(clique duas vezes para escrever, antes de rodar a próxima célula)*

In [ ]:
F2 = MF[:, [0, 2]]
dif = []
for s in range(10):
    a, b, ya, yb = train_test_split(F2, y, test_size=0.3, stratify=y, random_state=s)
    sc_ = StandardScaler().fit(a); a, b = sc_.transform(a), sc_.transform(b)
    dif.append(KNeighborsClassifier(11).fit(a, ya).score(b, yb) - KNeighborsClassifier(1).fit(a, ya).score(b, yb))
dif = np.array(dif)
v, e = np.sum(dif > 0), np.sum(dif == 0)
print(f'k = 11 venceu {v} vezes; {e} empates; diferença média = {dif.mean():+.1%} ± {dif.std():.1%}')
p_valor(v, 10 - e)

**Para discutir:** o que esse p-valor permite concluir, e o que ele **não** permite? Como esse raciocínio pode ser usado para comparar duas representações em vários níveis de uma degradação?